# 12 · GSE232381 · bulk_RNA_seq · normalise

Reads the counts, NCBI gene_info, `metadata.rds`. Writes `data/run_artifacts/GSE232381/expression.rds`.

1. **Entrez → symbol** with NCBI gene_info, the same names as GSE65391.
2. **Filter low counts** (WGCNA FAQ): remove genes with a count below 10 in more than 90% of samples.
3. **PFlog1pPF** (Booeshaghi AS, Hallgrímsdóttir IB, Gálvez-Merchán Á, Pachter L. "Depth
   normalization for single-cell genomics count data", bioRxiv 2022, doi:10.1101/2022.05.06.490859):
   - PF: scale each sample's counts so its total equals the mean total over samples;
   - log1p: log(1 + x);
   - PF again on the log values.

In [1]:
source("../src/paths.R")
meta <- readRDS(art("GSE232381", "metadata.rds"))
cnt <- read.delim(raw("GSE232381", "GSE232381_raw_counts_GRCh38.p13_NCBI.tsv.gz"), check.names = FALSE)
counts <- as.matrix(cnt[, meta$sample]); rownames(counts) <- cnt$GeneID
stopifnot(all(counts == round(counts)), all(counts >= 0))
dim(counts)

[1] 39376    16

In [2]:
gi  <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
sym <- gi$Symbol[match(rownames(counts), gi$GeneID)]
c(genes = nrow(counts), with_symbol = sum(!is.na(sym)), duplicated_symbols = sum(duplicated(na.omit(sym))))
counts <- counts[!is.na(sym), ]; sym <- sym[!is.na(sym)]

genes        with_symbol duplicated_symbols 
             39376              37663                  1

**Check.** A symbol shared by two Entrez identifiers: their counts are summed into one row.

In [3]:
dup <- sym[duplicated(sym)]
data.frame(symbol = sym[sym %in% dup], entrez = rownames(counts)[sym %in% dup], total_counts = rowSums(counts[sym %in% dup, , drop = FALSE]))
counts <- rowsum(counts, sym)
c(genes = nrow(counts), unique_names = length(unique(rownames(counts))))

,symbol,entrez,total_counts
,<chr>,<chr>,<dbl>
107985614,TRNAV-CAC,107985614,3
107985615,TRNAV-CAC,107985615,11


genes unique_names 
       37662        37662

**Result.** 37,663 of 39,376 Entrez identifiers have a symbol. One symbol, TRNAV-CAC, is shared by two
identifiers (3 and 11 counts in total); they are summed, leaving 37,662 genes.

**Library sizes** (total counts per sample).

In [4]:
data.frame(sample = colnames(counts), ln_active = meta$ln_active, total_counts_millions = round(colSums(counts) / 1e6, 1))

,sample,ln_active,total_counts_millions
,<chr>,<dbl>,<dbl>
GSM7329680,GSM7329680,1,25.5
GSM7329682,GSM7329682,1,14.8
GSM7329683,GSM7329683,1,28.0
GSM7329684,GSM7329684,1,28.5
GSM7329685,GSM7329685,1,11.7
GSM7329686,GSM7329686,1,26.4
GSM7329687,GSM7329687,1,22.9
GSM7329688,GSM7329688,1,13.6
GSM7329689,GSM7329689,1,25.6


In [5]:
keep <- rowMeans(counts < 10) <= 0.90
c(genes_before = nrow(counts), genes_kept = sum(keep))
counts <- counts[keep, ]

genes_before   genes_kept 
       37662        16605

**Result.** 16,605 genes pass the filter.

In [6]:
pf <- function(X) t(t(X) / colSums(X) * mean(colSums(X)))
E  <- pf(log1p(pf(counts)))
round(quantile(E), 2)

0%   25%   50%   75%  100% 
 0.00  3.08  4.66  5.89 18.62

**Check.** After PFlog1pPF every sample has the same total.

In [7]:
range(colSums(E))
saveRDS(list(E = E, counts = counts, meta = meta), art("GSE232381", "expression.rds"))

[1] 75262.76 75262.76